In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, hashlib, subprocess, time
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PARENT_DIR   = DRIVE_ROOT / 'CALSHIFT_Research'
PROJECT_ROOT = PARENT_DIR / 'calshift-research'
CRED_DIR     = DRIVE_ROOT / '.gitcreds'

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'], check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'], check=False)
subprocess.run(['git','config','--global','credential.helper','store'], check=False)

for fn, dest in [('.git-credentials','/root/.git-credentials'),
                 ('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR / fn, CRED_DIR / fn):
        if cand.exists():
            shutil.copy(cand, dest); os.chmod(dest, 0o600); break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
subprocess.run(['git','pull','--ff-only','--quiet'], check=False)

import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
git credentials restored from /content/drive/MyDrive/.gitcreds/.git-credentials
root: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - ladder parameters (Amendment 1 A3.2, Amendment 2 B5)
# =============================================================================
RUNGS          = [0.00, 0.20, 0.40, 0.60, 0.80]
D_EVAL_SIZE    = 2340
N_REALIZATIONS = config.N_LADDER_REALIZATIONS['nslkdd']   # 20
LADDER_SEED    = 20260725

# classes exempt from the unseen-fraction constraint
#   Normal : no unseen subtypes exist
#   U2R    : excluded from the ladder constraint (Amendment 1 A3.3)
EXEMPT = {'Normal', 'U2R'}

S_COV_PER_SIDE = 2000          # Amendment 2 B5 (preregistered 20000 unattainable)
S_COV_FOLDS    = 5
NULL_DRAWS     = config.PERMUTATION_NULL_DRAWS   # 200
NULL_Q         = config.PERMUTATION_NULL_Q       # 0.95

print(dict(rungs=RUNGS, d_eval=D_EVAL_SIZE, realizations=N_REALIZATIONS,
           exempt=sorted(EXEMPT), s_cov_per_side=S_COV_PER_SIDE))

{'rungs': [0.0, 0.2, 0.4, 0.6, 0.8], 'd_eval': 2340, 'realizations': 20, 'exempt': ['Normal', 'U2R'], 's_cov_per_side': 2000}


In [3]:
# =============================================================================
# Cell 3 - load partitions from notebook 02
# =============================================================================
nsl_train = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_train.parquet').reset_index(drop=True)
nsl_test  = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_test.parquet').reset_index(drop=True)
part      = pd.read_parquet(config.PROC_DIR / 'nslkdd_source_partition_labels.parquet')
nsl_train = nsl_train.assign(partition=part['partition'].values)

S_pool = nsl_train[nsl_train.partition == 'source_cal_pool']

seen_subtypes = set(nsl_train['subtype'])
Spool_subtypes = set(S_pool['subtype'])
nsl_test = nsl_test.assign(is_unseen=~nsl_test['subtype'].isin(seen_subtypes))

focal = json.loads((config.REPORTS_DIR / 'focal_class_record.json').read_text())
FOCAL_CLASS = focal['focal_class']

fp_path = config.REPORTS_DIR / 'partition_fingerprints.json'
if fp_path.exists():
    want = json.loads(fp_path.read_text())['fingerprints']
    got = hashlib.sha256(np.sort(S_pool.index.to_numpy().astype(np.int64)).tobytes()).hexdigest()
    ok = got == want['source/source_cal_pool']['sha256']
    print('partition matches notebook 02 fingerprint:', ok)
    assert ok, 'S_pool differs from the partition recorded in notebook 02'
else:
    print('WARNING: no partition fingerprint found; cannot verify against notebook 02')

print('S_pool:', len(S_pool), '| target pool:', len(nsl_test))
print('focal class:', FOCAL_CLASS)
print('unseen mass in target pool:', round(nsl_test['is_unseen'].mean(), 4))

partition matches notebook 02 fingerprint: True
S_pool: 18894 | target pool: 22544
focal class: R2L
unseen mass in target pool: 0.1663


In [4]:
# =============================================================================
# Cell 4 - per-class quotas at natural target prevalence
# =============================================================================
prev  = nsl_test['label'].value_counts(normalize=True)
QUOTA = (prev * D_EVAL_SIZE).round().astype(int).reindex(config.CANONICAL_CLASSES).fillna(0).astype(int)

avail_u = nsl_test[nsl_test.is_unseen].groupby('label').size().reindex(config.CANONICAL_CLASSES).fillna(0).astype(int)
avail_s = nsl_test[~nsl_test.is_unseen].groupby('label').size().reindex(config.CANONICAL_CLASSES).fillna(0).astype(int)

print('quota per rung (D_eval):', QUOTA.to_dict(), '| sum', int(QUOTA.sum()))
print()
ok_all = True
for cls in config.CANONICAL_CLASSES:
    if cls in EXEMPT: continue
    line = f'  {cls:6s}'
    for f in RUNGS:
        nu = int(round(QUOTA[cls] * f)); ns = QUOTA[cls] - nu
        ok = (2*nu <= avail_u[cls]) and (2*ns <= avail_s[cls])
        ok_all &= ok
        line += f'  f={f:.1f}:{"OK" if ok else "FAIL"}'
    print(line)
assert ok_all, 'ladder infeasible at some rung'
print('\nall rungs feasible for both D_eval and T_cal')

quota per rung (D_eval): {'Normal': 1008, 'DoS': 774, 'Probe': 251, 'R2L': 299, 'U2R': 7} | sum 2339

  DoS     f=0.0:OK  f=0.2:OK  f=0.4:OK  f=0.6:OK  f=0.8:OK
  Probe   f=0.0:OK  f=0.2:OK  f=0.4:OK  f=0.6:OK  f=0.8:OK
  R2L     f=0.0:OK  f=0.2:OK  f=0.4:OK  f=0.6:OK  f=0.8:OK

all rungs feasible for both D_eval and T_cal


In [5]:
# =============================================================================
# Cell 5 - ladder construction
# For each class, draw 2 x quota rows (D_eval half + disjoint T_cal half).
# Unseen quota is filled by taking unseen subtypes in a RANDOM order until the
# requirement is met, which is what varies across realizations. At high rungs
# the largest subtype becomes unavoidable (Amendment 1 A3.4).
# =============================================================================
def build_instance(rung, realization):
    rng = np.random.default_rng([LADDER_SEED, int(rung*100), realization])
    eval_idx, tcal_idx, used_subtypes = [], [], {}

    for cls in config.CANONICAL_CLASSES:
        q = int(QUOTA[cls])
        if q == 0: continue
        pool_c = nsl_test[nsl_test.label == cls]

        if cls in EXEMPT:
            take = rng.choice(pool_c.index.to_numpy(), size=min(2*q, len(pool_c)), replace=False)
            picked = list(take); used_subtypes[cls] = ['<exempt>']
        else:
            n_u = int(round(q * rung)); n_s = q - n_u
            picked = []

            if n_u > 0:
                subs = pool_c[pool_c.is_unseen].groupby('subtype').size()
                order = rng.permutation(subs.index.to_numpy())
                chosen, cum = [], 0
                for s in order:
                    chosen.append(s); cum += int(subs[s])
                    if cum >= 2*n_u: break
                assert cum >= 2*n_u, f'{cls} rung {rung}: unseen quota unreachable'
                cand = pool_c[pool_c.is_unseen & pool_c.subtype.isin(chosen)].index.to_numpy()
                picked += list(rng.choice(cand, size=2*n_u, replace=False))
                used_subtypes[cls] = sorted(chosen)
            else:
                used_subtypes[cls] = []

            if n_s > 0:
                cand = pool_c[~pool_c.is_unseen].index.to_numpy()
                picked += list(rng.choice(cand, size=2*n_s, replace=False))

        # Split so that D_eval and T_cal have IDENTICAL subtype composition.
        # Pooling then cutting in half leaves a composition gap of up to 7pp,
        # which would give TSC its own uncontrolled shift and contaminate the
        # primary contrast. Interleave within each subtype block instead.
        picked = np.array(picked)
        ev_c, tc_c = [], []
        for st, grp in nsl_test.loc[picked].groupby('subtype'):
            arr = grp.index.to_numpy().copy(); rng.shuffle(arr)
            ev_c += list(arr[0::2]); tc_c += list(arr[1::2])
        k = min(len(ev_c), len(tc_c))
        eval_idx += ev_c[:k]; tcal_idx += tc_c[:k]

    return np.array(sorted(eval_idx)), np.array(sorted(tcal_idx)), used_subtypes

e, t, u = build_instance(0.80, 0)
print('smoke test rung 0.80 realization 0')
print('  D_eval', len(e), '| T_cal', len(t), '| disjoint', len(set(e) & set(t)) == 0)
print('  unseen fraction  D_eval:', round(nsl_test.loc[e, "is_unseen"].mean(), 4),
      '| T_cal:', round(nsl_test.loc[t, "is_unseen"].mean(), 4))
gap = abs(nsl_test.loc[e, "is_unseen"].mean() - nsl_test.loc[t, "is_unseen"].mean())
print('  composition gap:', round(float(gap), 5))
assert gap < 0.01, 'D_eval and T_cal composition mismatch'
print('  subtypes used:', {k: v for k, v in u.items() if v and v != ["<exempt>"]})

smoke test rung 0.80 realization 0
  D_eval 2331 | T_cal 2331 | disjoint True
  unseen fraction  D_eval: 0.453 | T_cal: 0.4535
  composition gap: 0.00043
  subtypes used: {'DoS': ['apache2', 'processtable', 'udpstorm', 'worm'], 'Probe': ['mscan', 'saint'], 'R2L': ['httptunnel', 'named', 'sendmail', 'snmpgetattack', 'snmpguess', 'xlock', 'xsnoop']}


In [6]:
# =============================================================================
# Cell 6 - build all rung x realization instances
# =============================================================================
records, assignments = [], []
t0 = time.time()
for rung in RUNGS:
    for j in range(N_REALIZATIONS):
        e, t, used = build_instance(rung, j)
        assert len(set(e) & set(t)) == 0, 'D_eval and T_cal overlap'
        ce = nsl_test.loc[e]['label'].value_counts()
        ct = nsl_test.loc[t]['label'].value_counts()
        assert ce.reindex(ct.index).equals(ct), 'class counts differ between D_eval and T_cal'
        g = abs(nsl_test.loc[e, 'is_unseen'].mean() - nsl_test.loc[t, 'is_unseen'].mean())
        assert g < 0.01, f'composition gap {g:.4f} at rung {rung} realization {j}'
        for role, idx in [('eval', e), ('tcal', t)]:
            assignments.append(pd.DataFrame({'rung': rung, 'realization': j,
                                             'role': role, 'test_idx': idx}))
        sub = nsl_test.loc[e]
        records.append({
            'rung': rung, 'realization': j,
            'n_eval': len(e), 'n_tcal': len(t),
            'unseen_frac_eval': float(sub['is_unseen'].mean()),
            'unseen_frac_focal': float(sub[sub.label == FOCAL_CLASS]['is_unseen'].mean())
                                  if (sub.label == FOCAL_CLASS).any() else np.nan,
            'subtypes_used': json.dumps({k: v for k, v in used.items() if v}),
        })

ladder = pd.DataFrame(records)
assign = pd.concat(assignments, ignore_index=True)
print(f'built {len(ladder)} instances in {time.time()-t0:.1f}s')
print(ladder.groupby('rung')[['unseen_frac_eval','unseen_frac_focal']].mean().round(4).to_string())

built 100 instances in 6.5s
      unseen_frac_eval  unseen_frac_focal
rung                                     
0.0             0.0007             0.0000
0.2             0.1139             0.1991
0.4             0.2273             0.4008
0.6             0.3403             0.5985
0.8             0.4542             0.7986


In [7]:
# =============================================================================
# Cell 7 - feature matrix for the domain classifier
# Features only. Label and subtype are never exposed to the classifier.
# =============================================================================
FEATURE_COLS = [c for c in nsl_train.columns
                if c not in ('label', 'subtype', 'partition', 'is_unseen')]
CAT_COLS = ['protocol_type', 'service', 'flag']

cats = {c: pd.Categorical(pd.concat([nsl_train[c], nsl_test[c]]).astype(str)).categories
        for c in CAT_COLS}

def encode(df):
    X = df[FEATURE_COLS].copy()
    for c in CAT_COLS:
        X[c] = pd.Categorical(X[c].astype(str), categories=cats[c]).codes
    return X.astype(np.float32).to_numpy()

X_S = encode(S_pool)
print('feature columns:', len(FEATURE_COLS), '| S_pool matrix', X_S.shape)
assert 'label' not in FEATURE_COLS and 'subtype' not in FEATURE_COLS

feature columns: 41 | S_pool matrix (18894, 41)


In [8]:
# =============================================================================
# Cell 8 - S_cov: cross-fitted domain-classifier AUC
# =============================================================================
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def domain_auc(Xa, Xb, seed, per_side=S_COV_PER_SIDE, folds=S_COV_FOLDS):
    rng = np.random.default_rng(seed)
    na, nb = min(per_side, len(Xa)), min(per_side, len(Xb))
    a = Xa[rng.choice(len(Xa), na, replace=False)]
    b = Xb[rng.choice(len(Xb), nb, replace=False)]
    X = np.vstack([a, b])
    y = np.r_[np.zeros(na), np.ones(nb)]
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=int(seed) % (2**31))
    aucs = []
    for tr_i, te_i in skf.split(X, y):
        clf = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1,
                                             max_depth=6, random_state=0)
        clf.fit(X[tr_i], y[tr_i])
        aucs.append(roc_auc_score(y[te_i], clf.predict_proba(X[te_i])[:, 1]))
    return float(np.mean(aucs))

t0 = time.time()
scov = []
for i, r in ladder.iterrows():
    e_idx = assign[(assign.rung == r.rung) & (assign.realization == r.realization) &
                   (assign.role == 'eval')]['test_idx'].to_numpy()
    X_E = encode(nsl_test.loc[e_idx])
    scov.append(domain_auc(X_S, X_E, seed=1000 + i))
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(ladder)}  {time.time()-t0:.0f}s')
ladder['S_cov'] = scov
print(f'done in {time.time()-t0:.0f}s')
print(ladder.groupby('rung')['S_cov'].agg(['mean','std','min','max']).round(4).to_string())

  20/100  37s
  40/100  78s
  60/100  117s
  80/100  171s
  100/100  213s
done in 213s
        mean     std     min     max
rung                                
0.0   0.8485  0.0056  0.8403  0.8598
0.2   0.8571  0.0068  0.8419  0.8657
0.4   0.8638  0.0062  0.8495  0.8761
0.6   0.8750  0.0059  0.8639  0.8837
0.8   0.8927  0.0056  0.8848  0.9031


In [9]:
# =============================================================================
# Cell 9 - permutation null for S_cov
# Split S_pool at random, compute the same statistic. Anything below the 95th
# percentile of this distribution is indistinguishable from no shift.
# =============================================================================
t0 = time.time()
null = []
for d in range(NULL_DRAWS):
    rng = np.random.default_rng(50000 + d)
    perm = rng.permutation(len(X_S)); half = len(perm) // 2
    null.append(domain_auc(X_S[perm[:half]], X_S[perm[half:2*half]], seed=60000 + d))
    if (d + 1) % 50 == 0:
        print(f'  {d+1}/{NULL_DRAWS}  {time.time()-t0:.0f}s')

null = np.array(null)
S_COV_NULL_THRESHOLD = float(np.quantile(null, NULL_Q))
print(f'\nnull mean {null.mean():.4f} sd {null.std():.4f} '
      f'| {NULL_Q:.0%} threshold {S_COV_NULL_THRESHOLD:.4f}')

ladder['above_null'] = ladder['S_cov'] > S_COV_NULL_THRESHOLD
print('\ninstances above the no-shift threshold, by rung:')
print(ladder.groupby('rung')['above_null'].mean().round(3).to_string())
np.save(config.PROC_DIR / 'scov_permutation_null.npy', null)

  50/200  83s
  100/200  167s
  150/200  248s
  200/200  331s

null mean 0.5019 sd 0.0121 | 95% threshold 0.5202

instances above the no-shift threshold, by rung:
rung
0.0    1.0
0.2    1.0
0.4    1.0
0.6    1.0
0.8    1.0


In [10]:
# =============================================================================
# Cell 10 - S_lab and S_sup
#   S_lab = total variation between S_pool and D_eval class priors
#   S_sup = D_eval mass in subtypes absent from S_pool
# S_lab is a PLACEBO predictor for the class-conditional model (Amendment 2 B2.3)
# =============================================================================
p_cal = S_pool['label'].value_counts(normalize=True).reindex(config.CANONICAL_CLASSES).fillna(0)

slab, ssup, ssup_focal = [], [], []
for _, r in ladder.iterrows():
    e_idx = assign[(assign.rung == r.rung) & (assign.realization == r.realization) &
                   (assign.role == 'eval')]['test_idx'].to_numpy()
    ev = nsl_test.loc[e_idx]
    p_ev = ev['label'].value_counts(normalize=True).reindex(config.CANONICAL_CLASSES).fillna(0)
    slab.append(float(0.5 * np.abs(p_cal - p_ev).sum()))
    ssup.append(float((~ev['subtype'].isin(Spool_subtypes)).mean()))
    f = ev[ev.label == FOCAL_CLASS]
    ssup_focal.append(float((~f['subtype'].isin(Spool_subtypes)).mean()) if len(f) else np.nan)

ladder['S_lab'] = slab
ladder['S_sup'] = ssup
ladder['S_sup_focal'] = ssup_focal
print(ladder.groupby('rung')[['S_cov','S_lab','S_sup','S_sup_focal']]
      .mean().round(4).to_string())

if ladder['S_lab'].std() < 1e-9:
    print('\nNOTE: S_lab has zero variance within this dataset because class '
          'prevalence is held fixed by design. It is NOT estimable here and must '
          'be omitted from the NSL-KDD model. It becomes estimable only when '
          'datasets with differing prevalence enter the pooled model.')

       S_cov   S_lab   S_sup  S_sup_focal
rung                                     
0.0   0.8485  0.1363  0.0010       0.0015
0.2   0.8571  0.1361  0.1142       0.2001
0.4   0.8638  0.1360  0.2276       0.4022
0.6   0.8750  0.1361  0.3407       0.5998
0.8   0.8927  0.1360  0.4543       0.7986


In [11]:
# =============================================================================
# Cell 11 - Arm B weights (marginal coverage only, Amendment 2 B2.1)
# Recorded here; applied in the marginal analysis. Class-conditional coverage
# is invariant to these weights by construction.
# =============================================================================
armb = []
for _, r in ladder.iterrows():
    e_idx = assign[(assign.rung == r.rung) & (assign.realization == r.realization) &
                   (assign.role == 'eval')]['test_idx'].to_numpy()
    ev = nsl_test.loc[e_idx]
    p_ev = ev['label'].value_counts(normalize=True).reindex(config.CANONICAL_CLASSES).fillna(0)
    w = (p_cal / p_ev.replace(0, np.nan)).fillna(0)
    wi = ev['label'].map(w).to_numpy(dtype=float)
    ess = float(wi.sum()**2 / np.square(wi).sum())
    armb.append({'rung': r.rung, 'realization': r.realization,
                 'ess_marginal': ess, 'n_eval': len(ev),
                 **{f'w_{c}': float(w[c]) for c in config.CANONICAL_CLASSES}})

armb = pd.DataFrame(armb)
armb.to_csv(config.REPORTS_DIR / 'armb_weights_nslkdd.csv', index=False)
print(armb.groupby('rung')[['ess_marginal','n_eval']].mean().round(1).to_string())
print('\nmean weights by class:')
print(armb[[c for c in armb.columns if c.startswith('w_')]].mean().round(4).to_string())

      ess_marginal  n_eval
rung                      
0.0         2040.4  2333.8
0.2         2039.9  2332.6
0.4         2039.8  2332.2
0.6         2039.9  2332.3
0.8         2039.4  2331.9

mean weights by class:
w_Normal    1.2371
w_DoS       1.1016
w_Probe     0.8634
w_R2L       0.0619
w_U2R       0.1597


In [12]:
# =============================================================================
# Cell 12 - persist ladder and manifest
# =============================================================================
assign.to_parquet(config.PROC_DIR / 'nslkdd_ladder_assignments.parquet', index=False)
ladder.to_csv(config.REPORTS_DIR / 'ladder_shift_measures_nslkdd.csv', index=False)

def fp(idx):
    return hashlib.sha256(np.sort(np.asarray(idx, dtype=np.int64)).tobytes()).hexdigest()

fps = {}
for (rung, j, role), g in assign.groupby(['rung', 'realization', 'role']):
    fps[f'{rung:.2f}/{j}/{role}'] = fp(g['test_idx'])

manifest = {
    'ladder_seed': LADDER_SEED, 'rungs': RUNGS,
    'n_realizations': N_REALIZATIONS, 'd_eval_size': D_EVAL_SIZE,
    'quota_per_class': {k: int(v) for k, v in QUOTA.items()},
    'exempt_classes': sorted(EXEMPT), 'focal_class': FOCAL_CLASS,
    's_cov_per_side': S_COV_PER_SIDE, 's_cov_folds': S_COV_FOLDS,
    's_cov_null_draws': NULL_DRAWS,
    's_cov_null_threshold': S_COV_NULL_THRESHOLD,
    'n_instance_fingerprints': len(fps),
    'fingerprints': fps,
}
(config.REPORTS_DIR / 'ladder_manifest_nslkdd.json').write_text(json.dumps(manifest, indent=2))

dev = ('\n## notebook 03\n'
       '- S_cov subsample reduced from the preregistered 20000 per side to '
       f'{S_COV_PER_SIDE}, because Amendment 1 fixed D_eval at {D_EVAL_SIZE} rows '
       'and 20000 per side is unattainable on the evaluation side. '
       'Recorded in Amendment 2 B5.\n')
with open(config.REPORTS_DIR / 'deviations.md', 'a') as f:
    f.write(dev)

print(json.dumps({k: v for k, v in manifest.items() if k != 'fingerprints'}, indent=2))

{
  "ladder_seed": 20260725,
  "rungs": [
    0.0,
    0.2,
    0.4,
    0.6,
    0.8
  ],
  "n_realizations": 20,
  "d_eval_size": 2340,
  "quota_per_class": {
    "Normal": 1008,
    "DoS": 774,
    "Probe": 251,
    "R2L": 299,
    "U2R": 7
  },
  "exempt_classes": [
    "Normal",
    "U2R"
  ],
  "focal_class": "R2L",
  "s_cov_per_side": 2000,
  "s_cov_folds": 5,
  "s_cov_null_draws": 200,
  "s_cov_null_threshold": 0.5201770625000001,
  "n_instance_fingerprints": 200
}


In [13]:
def git(*args, show=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if show:
        if r.stdout.strip(): print(r.stdout.strip())
        if r.stderr.strip(): print(r.stderr.strip())
    return r

for s, d in [('/root/.git-credentials', PARENT_DIR / '.git-credentials'),
             ('/root/.gitconfig',       PARENT_DIR / '.gitconfig')]:
    if os.path.exists(s): shutil.copy(s, d)

os.chdir(PROJECT_ROOT)
git('add','-A', show=False)
if git('status','--porcelain', show=False).stdout.strip():
    git('commit','-m','nb03: ladders and shift measures')
    r = git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else:
    print('nothing to commit')
print(git('log','--oneline','-3', show=False).stdout)

[main 9fa0aaa] nb03: shift ladders, S_cov with permutation null, S_lab, S_sup, Arm B weights
 6 files changed, 438 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/03_ladders_and_shift.ipynb
 create mode 100644 reports/armb_weights_nslkdd.csv
 create mode 100644 reports/deviations.md
 create mode 100644 reports/ladder_manifest_nslkdd.json
 create mode 100644 reports/ladder_shift_measures_nslkdd.csv
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://github.com/anasbiswas1/calshift-research.git
   921eae4..9fa0aaa  main -> main
push exit: 0
9fa0aaa nb03: shift ladders, S_cov with permutation null, S_lab, S_sup, Arm B weights
921eae4 notebooks: credential discovery, non-raising git
8b12467 nb02: source partitions, binding feasibility table, focal class record
ddca18b nb02: source partitions, binding feasibility table, focal class record
2458ca9 nb01: setup, NSL-KDD and corrected CIC-IDS2017 acquisition, manifest

